# Fair Comparison: Token Pruning & Reduction Methods

Evaluates all methods on the **same test set**, with the **same metrics**:
- F1 Macro, Accuracy, TAR@FAR(1e-4)
- Latency (ms/image) — wall-clock, batch=1 on GPU
- GFLOPs/image

Methods:
1. **Baseline** — no pruning
2. **Random** — random token selection
3. **CLS Attention** — prune by raw CLS attention at the prune layer
4. **Forecaster (ours)** — AttentionForecaster predicts layer-23 attention from layer-2
5. **Forecaster + FT (ours)** — same, but classifier fine-tuned with pruning
6. **CropR** — learned token pruning baseline
7. **ToMe** — token merging (r-sweep)


## 0 · Imports & Reproducibility

In [ ]:
import copy
import json
import time
import types
import sys
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import timm
from peft import LoraConfig
from peft.tuners.lora import LoraModel
from sklearn.metrics import f1_score
from tome.patch import timm as tome_patch_timm
from fvcore.nn import FlopCountAnalysis

sys.path.append(".")
from src.dataset import HistologicalImageDataset
from src.vision_transformer_copr import VisionTransformer as CroprVisionTransformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | timm: {timm.__version__}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 1 · Configuration

In [ ]:
DATASET_NAME = "BREAKHIS"   # change to switch dataset

CFG = dict(
    data_dir    = f"/data/{DATASET_NAME}",
    img_size    = 224,
    patch_size  = 16,
    embed_dim   = 1024,
    # Evaluation
    batch_size  = 32,    # for throughput measurement
    num_workers = 8,
    far_threshold = 1e-4,
    seed        = SEED,
    # Pruning
    prune_layer  = 2,
    target_layer = 23,
    keep_ratios  = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0],
    # ToMe r values (tokens merged per block)
    tome_r_values = [0, 4, 8, 16, 24, 32],
    # Checkpoints
    ckpt_base = Path(f"/data/checkpoints-Attention-Pruning/{DATASET_NAME}"),
    results_dir = Path(f"results/{DATASET_NAME}/fair_comparison"),
)

CFG["ckpt_classifier"]      = CFG["ckpt_base"] / "uni_finetuned"   / "best_model.pt"
CFG["ckpt_forecaster"]      = CFG["ckpt_base"] / "forecaster"       / "forecaster_src02_tgt23.pt"
CFG["ckpt_pruned_ft"]       = CFG["ckpt_base"] / "pruned_finetuned" / "best_prune_layer2_keep10.pt"
CFG["ckpt_cropr"]           = CFG["ckpt_base"] / "uni_cropr"        / "best_model.pt"
CFG["num_patches"]          = (CFG["img_size"] // CFG["patch_size"]) ** 2  # 196

CFG["results_dir"].mkdir(parents=True, exist_ok=True)
print("Config:")
for k, v in CFG.items():
    print(f"  {k:22s}: {v}")

## 2 · Dataset

In [ ]:
eval_tf = T.Compose([
    T.Resize((CFG["img_size"], CFG["img_size"])),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

test_ds = HistologicalImageDataset(f"{CFG['data_dir']}/test", transform=eval_tf)

# Two loaders: one with batch_size=1 for latency, one large for F1/ACC
loader_eval = DataLoader(
    test_ds, batch_size=CFG["batch_size"], shuffle=False,
    num_workers=CFG["num_workers"], pin_memory=True, persistent_workers=True,
)
loader_latency = DataLoader(
    test_ds, batch_size=1, shuffle=False,
    num_workers=2, pin_memory=True, persistent_workers=True,
)

CLASS_NAMES = test_ds.class_names
N_CLASSES   = len(CLASS_NAMES)
print(f"Test: {len(test_ds)} images | Classes ({N_CLASSES}): {CLASS_NAMES}")

## 3 · Model Architectures

In [ ]:
# ─── 3.1 Base LoRA classifier ──────────────────────────────────────────────
class UNILoRAClassifier(nn.Module):
    def __init__(self, n_classes, dropout=0.1):
        super().__init__()
        backbone = timm.create_model(
            "hf-hub:MahmoodLab/uni", pretrained=True,
            init_values=1e-5, dynamic_img_size=True, num_classes=0,
        )
        lora_cfg = LoraConfig(
            r=8, lora_alpha=32,
            target_modules=["qkv", "proj", "fc1", "fc2"],
            lora_dropout=0.1, bias="none",
        )
        self.backbone = LoraModel(backbone, lora_cfg, adapter_name="default")
        self.head = nn.Sequential(
            nn.LayerNorm(1024), nn.Dropout(dropout), nn.Linear(1024, n_classes)
        )

    def forward(self, x):
        return self.head(self.backbone(x))


# ─── 3.2 Attention Forecaster ──────────────────────────────────────────────
class AttentionForecaster(nn.Module):
    def __init__(self, embed_dim=1024, hidden=256, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.input_proj  = nn.Linear(embed_dim, hidden)
        self.cls_query   = nn.Parameter(torch.randn(1, 1, hidden) * 0.02)
        self.self_attn   = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=hidden, nhead=n_heads, dim_feedforward=hidden * 2,
                dropout=dropout, batch_first=True, norm_first=True
            ) for _ in range(n_layers)
        ])
        self.cross_attn  = nn.ModuleList([
            nn.MultiheadAttention(hidden, n_heads, dropout=dropout, batch_first=True)
            for _ in range(n_layers)
        ])
        self.cross_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_layers)])
        self.norm        = nn.LayerNorm(hidden)
        self.score_head  = nn.Sequential(
            nn.Linear(hidden * 2, 128), nn.GELU(), nn.Dropout(dropout), nn.Linear(128, 1)
        )

    def forward(self, x):
        B, N, D = x.shape
        x = self.input_proj(x)
        for sa in self.self_attn:
            x = sa(x)
        cls = self.cls_query.expand(B, -1, -1)
        for ca, norm in zip(self.cross_attn, self.cross_norms):
            cls_out, _ = ca(cls, x, x)
            cls = norm(cls + cls_out)
        x_norm  = self.norm(x)
        cls_exp = cls.expand(-1, N, -1)
        return self.score_head(torch.cat([x_norm, cls_exp], dim=-1)).squeeze(-1).softmax(-1)


# ─── 3.3 CropR ─────────────────────────────────────────────────────────────
class UNICroprClassifier(CroprVisionTransformer):
    def __init__(self, cropr_cfg, init_values: float = 1e-5, **kwargs):
        super().__init__(cropr_cfg, **kwargs)
        dpr = [x.item() for x in torch.linspace(0, kwargs["drop_path_rate"], len(self.blocks))]
        if cropr_cfg["use_cropr"]:
            dpr[-1] = 0.0
        for i in range(len(self.blocks)):
            self.blocks[i] = timm.models.vision_transformer.Block(
                dim=self.embed_dim, num_heads=kwargs["num_heads"],
                qkv_bias=True, init_values=init_values,
                drop_path=dpr[i], norm_layer=partial(nn.LayerNorm, eps=1e-6),
            )


def build_cropr_model(n_classes):
    cropr_cfg = dict(
        use_cropr=True, pruning_rate=8, llf=False, num_queries=1,
        num_heads=1, pre_attn_norm=False, q_proj=False, k_proj=False,
        v_proj=False, mlp=True, mlp_ratio=4.0, training=True,
    )
    return UNICroprClassifier(
        cropr_cfg, init_values=1e-5,
        num_classes=n_classes, img_size=CFG["img_size"],
        patch_size=CFG["patch_size"], embed_dim=1024, depth=24,
        num_heads=16, mlp_ratio=4.0, drop_path_rate=0.2,
        global_pool="avg", class_token=True,
    )


print("Architecture classes defined.")

## 4 · Metric Helpers

In [ ]:
def compute_tar_at_far(scores, is_correct, far_threshold=1e-4):
    scores    = np.array(scores)
    correct   = np.array(is_correct, dtype=bool)
    incorrect = ~correct
    if incorrect.sum() == 0:
        return 1.0
    n_far     = max(1, int(np.ceil(incorrect.sum() * far_threshold)))
    threshold = np.sort(scores[incorrect])[::-1][min(n_far - 1, incorrect.sum() - 1)]
    return float((scores[correct] >= threshold).mean())


def compute_metrics(preds, labels, scores, far_threshold=1e-4):
    preds, labels, scores = map(np.array, (preds, labels, scores))
    f1  = f1_score(labels, preds, average="macro", zero_division=0)
    acc = float((preds == labels).mean())
    tar = compute_tar_at_far(scores, preds == labels, far_threshold)
    return {"accuracy": acc, "f1_macro": f1, "tar_at_far": tar}


@torch.no_grad()
def run_inference(model, loader, device):
    """Returns predictions, ground-truth labels, max-prob confidence."""
    model.eval()
    all_preds, all_labels, all_scores = [], [], []
    autocast_on = (device.type == "cuda")
    for imgs, labels in tqdm(loader, leave=False, desc="Eval"):
        imgs = imgs.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=autocast_on):
            logits = model(imgs)
        probs  = logits.float().softmax(-1).cpu()
        preds  = probs.argmax(-1)
        scores = probs.max(-1).values
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())
        all_scores.extend(scores.tolist())
    return all_preds, all_labels, all_scores


@torch.no_grad()
def measure_latency_ms(model, loader, n_warmup=20, device=device):
    """Wall-clock ms/image on GPU with batch_size=1."""
    model.eval()
    autocast_on = (device.type == "cuda")
    it = iter(loader)
    for _ in range(n_warmup):
        imgs, _ = next(it)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=autocast_on):
            _ = model(imgs.to(device))
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0, n_total = time.perf_counter(), 0
    for imgs, _ in loader:
        imgs = imgs.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=autocast_on):
            _ = model(imgs)
        if device.type == "cuda":
            torch.cuda.synchronize()
        n_total += len(imgs)
    return (time.perf_counter() - t0) / n_total * 1000


def measure_gflops(model, device=device, img_size=224):
    """GFLOPs for a single image. Returns None if fvcore fails."""
    model.eval()
    dummy = torch.randn(1, 3, img_size, img_size, device=device)
    try:
        fa = FlopCountAnalysis(model, dummy)
        fa.unsupported_ops_warnings(False)
        fa.uncalled_modules_warnings(False)
        return fa.total() / 1e9
    except Exception as e:
        print(f"[WARN] GFLOPs unavailable: {e}")
        return None


print("Metric helpers ready.")

## 5 · Checkpoint Loader

In [ ]:
def load_ckpt(model, path, strict=False):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    state = torch.load(path, map_location="cpu")
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    state = {k.replace("module.", ""): v for k, v in state.items()}
    missing, unexpected = model.load_state_dict(state, strict=strict)
    if missing:     print(f"  [WARN] {len(missing)} missing keys")
    if unexpected:  print(f"  [WARN] {len(unexpected)} unexpected keys")
    return model


def freeze(model):
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    return model


print("Loader ready.")

## 6 · Pruning-Aware Forward Pass

A single generic function handles **Random**, **CLS Attention**, and **Forecaster** scoring strategies — keeping inference identical except for the token selection criterion.

In [ ]:
def forward_with_pruning(model, imgs, device, prune_layer, keep_ratio, score_fn):
    """
    Injects a hook at `prune_layer` that:
      1. Captures patch embeddings and CLS attention.
      2. Calls `score_fn(emb, attn)` → importance scores (B, N_patches).
      3. Keeps top-k tokens; discards the rest.

    score_fn signature: (emb: Tensor B×N×D, attn: Tensor B×H×N×N | None) → Tensor B×N
    """
    imgs_dev = imgs.to(device, non_blocking=True)
    cache = {"emb": None, "attn": None}

    def make_hook(layer_idx):
        def fwd(self, x):
            B, N, C = x.shape
            qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
            q, k, v = qkv.unbind(0)
            q, k = self.q_norm(q), self.k_norm(k)
            attn = (q @ k.transpose(-2, -1) * self.scale).softmax(-1)

            if layer_idx == prune_layer:
                cache["emb"]  = x[:, 1:].detach()          # patch embeddings
                cache["attn"] = attn.detach()               # full attn map

            x = (self.attn_drop(attn) @ v).transpose(1, 2).reshape(B, N, C)
            return self.proj_drop(self.proj(x))
        return fwd

    def prune_hook(module, input, output):
        """Attached to the block (not attention) to prune AFTER the full block runs."""
        x = output
        B, N, _ = x.shape
        scores = score_fn(cache["emb"], cache["attn"])  # (B, N-1)
        k_keep = max(1, int((N - 1) * keep_ratio))
        topk_idx = scores.topk(k_keep, dim=-1).indices   # (B, k_keep)

        cls_tok = x[:, :1, :]
        patches = x[:, 1:, :]
        kept    = torch.stack([patches[b][topk_idx[b]] for b in range(B)])
        return torch.cat([cls_tok, kept], dim=1)

    # Attach hooks
    orig_attn_fwd = model.backbone.model.blocks[prune_layer].attn.forward
    model.backbone.model.blocks[prune_layer].attn.forward = types.MethodType(
        make_hook(prune_layer), model.backbone.model.blocks[prune_layer].attn
    )
    hook_handle = model.backbone.model.blocks[prune_layer].register_forward_hook(prune_hook)

    with torch.no_grad():
        logits = model(imgs_dev)

    # Restore
    model.backbone.model.blocks[prune_layer].attn.forward = orig_attn_fwd
    hook_handle.remove()

    return logits.float().cpu()


# ─── Scoring functions ──────────────────────────────────────────────────────
def score_random(emb, attn):
    B, N, _ = emb.shape
    return torch.rand(B, N)


def score_cls_attention(emb, attn):
    # Mean over heads of CLS-to-patch attention at the prune layer
    return attn[:, :, 0, 1:].mean(dim=1).cpu()  # (B, N_patches)


def make_forecaster_scorer(forecaster):
    def score_fn(emb, attn):
        with torch.no_grad():
            return forecaster(emb.to(next(forecaster.parameters()).device)).cpu()
    return score_fn


print("Pruning forward pass ready.")

## 7 · Evaluate a Single Method at All Keep Ratios

In [ ]:
def evaluate_pruning_method(
    method_name, model, loader_eval, loader_latency,
    keep_ratios, prune_layer, score_fn, device,
    far_threshold=1e-4,
):
    """
    For each keep_ratio, runs inference using forward_with_pruning and
    collects metrics + latency.
    """
    records = []
    for kr in tqdm(keep_ratios, desc=method_name):
        all_preds, all_labels, all_scores = [], [], []

        for imgs, labels in loader_eval:
            logits = forward_with_pruning(
                model, imgs, device,
                prune_layer=prune_layer,
                keep_ratio=kr,
                score_fn=score_fn,
            )
            probs  = logits.softmax(-1)
            preds  = probs.argmax(-1)
            scores = probs.max(-1).values
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())
            all_scores.extend(scores.tolist())

        metrics = compute_metrics(all_preds, all_labels, all_scores, far_threshold)

        # Latency: wrap forward_with_pruning in a callable model-like object
        class _PrunedModel(nn.Module):
            def __init__(self, m, pl, kr_, sf):
                super().__init__()
                self._m, self._pl, self._kr, self._sf = m, pl, kr_, sf
            def forward(self, x):
                return forward_with_pruning(self._m, x.cpu(), device, self._pl, self._kr, self._sf).to(x.device)

        ms = measure_latency_ms(_PrunedModel(model, prune_layer, kr, score_fn), loader_latency, device=device)

        records.append({
            "method":      method_name,
            "keep_ratio":  kr,
            "accuracy":    metrics["accuracy"],
            "f1_macro":    metrics["f1_macro"],
            "tar_at_far":  metrics["tar_at_far"],
            "ms_per_img":  ms,
            "gflops":      None,   # computed separately (depends on actual token count)
        })
        print(f"  keep={int(kr*100):3d}%  "
              f"ACC={metrics['accuracy']:.4f}  "
              f"F1={metrics['f1_macro']:.4f}  "
              f"TAR={metrics['tar_at_far']:.4f}  "
              f"ms={ms:.2f}")
    return records


print("evaluate_pruning_method ready.")

## 8 · Load Models

In [ ]:
# ── 8.1 Baseline classifier (no pruning) ────────────────────────────────────
print("Loading baseline...")
classifier = freeze(load_ckpt(
    UNILoRAClassifier(N_CLASSES).to(device),
    CFG["ckpt_classifier"], strict=False,
))

# ── 8.2 Forecaster ──────────────────────────────────────────────────────────
print("Loading forecaster...")
forecaster = freeze(load_ckpt(
    AttentionForecaster().to(device),
    CFG["ckpt_forecaster"], strict=True,
))

# ── 8.3 Fine-tuned-with-pruning classifier ───────────────────────────────────
print("Loading pruned-finetuned classifier...")
classifier_ft = freeze(load_ckpt(
    UNILoRAClassifier(N_CLASSES).to(device),
    CFG["ckpt_pruned_ft"], strict=False,
))

# ── 8.4 CropR ───────────────────────────────────────────────────────────────
print("Loading CropR...")
classifier_cropr = freeze(load_ckpt(
    build_cropr_model(N_CLASSES).to(device),
    CFG["ckpt_cropr"], strict=False,
))

print("\nAll models loaded ✓")

## 9 · Run All Methods

### 9.1 Baseline (No Pruning)

In [ ]:
print("=== BASELINE (No Pruning) ===")
preds_base, labels_base, scores_base = run_inference(classifier, loader_eval, device)
metrics_base = compute_metrics(preds_base, labels_base, scores_base, CFG["far_threshold"])
ms_base   = measure_latency_ms(classifier, loader_latency, device=device)
gflops_base = measure_gflops(classifier, device)

baseline_row = [{
    "method":     "Baseline (No Pruning)",
    "keep_ratio": 1.0,
    "accuracy":   metrics_base["accuracy"],
    "f1_macro":   metrics_base["f1_macro"],
    "tar_at_far": metrics_base["tar_at_far"],
    "ms_per_img": ms_base,
    "gflops":     gflops_base,
}]

print(f"  ACC={metrics_base['accuracy']:.4f}  "
      f"F1={metrics_base['f1_macro']:.4f}  "
      f"TAR={metrics_base['tar_at_far']:.4f}  "
      f"ms={ms_base:.2f}  GFLOPs={gflops_base:.2f}")

### 9.2 Pruning Methods (Random / CLS Attn / Forecaster / Forecaster+FT)

In [ ]:
pruning_methods = [
    ("Random",             classifier,    score_random),
    ("CLS Attention",      classifier,    score_cls_attention),
    ("Forecaster",         classifier,    make_forecaster_scorer(forecaster)),
    ("Forecaster + FT",    classifier_ft, make_forecaster_scorer(forecaster)),
]

all_pruning_records = []
for method_name, model, score_fn in pruning_methods:
    print(f"\n=== {method_name} ===")
    records = evaluate_pruning_method(
        method_name, model, loader_eval, loader_latency,
        keep_ratios=CFG["keep_ratios"],
        prune_layer=CFG["prune_layer"],
        score_fn=score_fn,
        device=device,
        far_threshold=CFG["far_threshold"],
    )
    all_pruning_records.extend(records)

### 9.3 CropR

In [ ]:
print("=== CropR ===")
preds_cropr, labels_cropr, scores_cropr = run_inference(classifier_cropr, loader_eval, device)
metrics_cropr = compute_metrics(preds_cropr, labels_cropr, scores_cropr, CFG["far_threshold"])
ms_cropr      = measure_latency_ms(classifier_cropr, loader_latency, device=device)
gflops_cropr  = measure_gflops(classifier_cropr, device)

cropr_row = [{
    "method":     "CropR",
    "keep_ratio": None,   # CropR has fixed pruning rate
    "accuracy":   metrics_cropr["accuracy"],
    "f1_macro":   metrics_cropr["f1_macro"],
    "tar_at_far": metrics_cropr["tar_at_far"],
    "ms_per_img": ms_cropr,
    "gflops":     gflops_cropr,
}]

print(f"  ACC={metrics_cropr['accuracy']:.4f}  "
      f"F1={metrics_cropr['f1_macro']:.4f}  "
      f"TAR={metrics_cropr['tar_at_far']:.4f}  "
      f"ms={ms_cropr:.2f}  GFLOPs={gflops_cropr:.2f}")

### 9.4 ToMe (Token Merging)

In [ ]:
print("=== ToMe ===")
tome_records = []
baseline_thr = 1.0  # will be filled at r=0

for r in tqdm(CFG["tome_r_values"], desc="ToMe r"):
    model_r = copy.deepcopy(classifier).to(device)

    if r > 0:
        # Apply ToMe to the inner ViT, not the PEFT wrapper
        tome_patch_timm(model_r.backbone.model, trace_source=False, prop_attn=True)
        model_r.backbone.model.r = r

    freeze(model_r)

    preds_t, labels_t, scores_t = run_inference(model_r, loader_eval, device)
    metrics_t = compute_metrics(preds_t, labels_t, scores_t, CFG["far_threshold"])
    ms_t      = measure_latency_ms(model_r, loader_latency, device=device)
    gflops_t  = measure_gflops(model_r, device)

    # Approximate keep_ratio equivalent from number of remaining tokens
    tokens_left = max(CFG["num_patches"] - r * 24, 1)
    keep_ratio_equiv = tokens_left / CFG["num_patches"]

    tome_records.append({
        "method":     f"ToMe",
        "keep_ratio": keep_ratio_equiv,
        "r":          r,
        "accuracy":   metrics_t["accuracy"],
        "f1_macro":   metrics_t["f1_macro"],
        "tar_at_far": metrics_t["tar_at_far"],
        "ms_per_img": ms_t,
        "gflops":     gflops_t,
    })
    print(f"  r={r:2d}  keep≈{keep_ratio_equiv:.2f}  "
          f"ACC={metrics_t['accuracy']:.4f}  "
          f"F1={metrics_t['f1_macro']:.4f}  "
          f"ms={ms_t:.2f}  GFLOPs={gflops_t:.2f}")

    del model_r
    torch.cuda.empty_cache()

## 10 · Assemble Results

In [ ]:
all_records = baseline_row + all_pruning_records + cropr_row + tome_records
df = pd.DataFrame(all_records)

# Save full CSV
csv_path = CFG["results_dir"] / "all_methods_results.csv"
df.to_csv(csv_path, index=False)
print(f"Results saved → {csv_path}")

# ── Pretty summary table at keep_ratio ≈ 0.1, 0.5, 1.0 ─────────────────────
key_krs = [0.1, 0.5, 1.0]
summary_rows = []

# Baseline
summary_rows.append(baseline_row[0])

# Pruning methods at key keep_ratios
for method in ["Random", "CLS Attention", "Forecaster", "Forecaster + FT"]:
    sub = df[df["method"] == method]
    for kr in key_krs:
        row = sub.iloc[(sub["keep_ratio"] - kr).abs().argmin()].to_dict()
        row["method"] = f"{method} (keep≈{int(kr*100)}%)"
        summary_rows.append(row)

# CropR
summary_rows.append(cropr_row[0])

# ToMe best (highest F1)
best_tome = max(tome_records, key=lambda x: x["f1_macro"])
best_tome["method"] = f"ToMe best (r={best_tome.get('r', '?')})"
summary_rows.append(best_tome)

df_summary = pd.DataFrame(summary_rows)[["method", "keep_ratio", "accuracy",
                                          "f1_macro", "tar_at_far", "ms_per_img", "gflops"]]
print("\n" + df_summary.to_string(index=False))

## 11 · Plots

In [ ]:
METHOD_STYLE = {
    "Random":           {"color": "#aaaaaa", "ls": ":",  "marker": "x",  "lw": 1.4},
    "CLS Attention":    {"color": "#4878cf", "ls": "--", "marker": "s",  "lw": 1.8},
    "Forecaster":       {"color": "#f4a261", "ls": "--", "marker": "^",  "lw": 1.8},
    "Forecaster + FT":  {"color": "#e63946", "ls": "-",  "marker": "o",  "lw": 2.5},
    "ToMe":             {"color": "#2a9d8f", "ls": "-.", "marker": "D",  "lw": 1.8},
}

BASELINE_F1  = baseline_row[0]["f1_macro"]
BASELINE_MS  = baseline_row[0]["ms_per_img"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), dpi=160)

for method_name, style in METHOD_STYLE.items():
    sub = df[df["method"] == method_name].sort_values("keep_ratio")
    if sub.empty:
        continue
    kw = dict(color=style["color"], ls=style["ls"],
              marker=style["marker"], lw=style["lw"],
              markersize=6, label=method_name)
    axes[0].plot(sub["keep_ratio"],  sub["f1_macro"],   **kw)
    axes[1].plot(sub["keep_ratio"],  sub["ms_per_img"], **kw)
    axes[2].plot(sub["ms_per_img"],  sub["f1_macro"],   **kw)

# Baseline reference lines
for ax in axes[:2]:
    ax.axhline(BASELINE_F1, color="#555", ls=":", lw=1.2, label="Baseline")
axes[2].axvline(BASELINE_MS, color="#555", ls=":", lw=1.2, label="Baseline")
axes[2].axhline(BASELINE_F1, color="#555", ls=":", lw=1.2)

# CropR as scatter
ax_scatter_args = dict(marker="*", s=200, zorder=5)
axes[0].scatter(cropr_row[0]["keep_ratio"] or 0.5,
                cropr_row[0]["f1_macro"], color="#6a0dad", label="CropR", **ax_scatter_args)
axes[1].scatter(cropr_row[0]["keep_ratio"] or 0.5,
                cropr_row[0]["ms_per_img"], color="#6a0dad", label="CropR", **ax_scatter_args)
axes[2].scatter(cropr_row[0]["ms_per_img"],
                cropr_row[0]["f1_macro"], color="#6a0dad", label="CropR", **ax_scatter_args)

axes[0].set_xlabel("Keep Ratio");     axes[0].set_ylabel("F1 Macro")
axes[1].set_xlabel("Keep Ratio");     axes[1].set_ylabel("Latency (ms/img)")
axes[2].set_xlabel("Latency (ms/img)"); axes[2].set_ylabel("F1 Macro")

titles = [
    f"F1 vs Keep Ratio ({DATASET_NAME})",
    f"Latency vs Keep Ratio ({DATASET_NAME})",
    f"Quality-Efficiency Tradeoff ({DATASET_NAME})",
]
for ax, title in zip(axes, titles):
    ax.set_title(title, fontsize=11)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8, loc="best")

plt.tight_layout()
fig.savefig(CFG["results_dir"] / "comparison_tradeoff.pdf", bbox_inches="tight")
fig.savefig(CFG["results_dir"] / "comparison_tradeoff.png", dpi=300, bbox_inches="tight")
plt.show()
print("Plots saved.")

### 11.1 TAR@FAR Plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), dpi=160)

for method_name, style in METHOD_STYLE.items():
    sub = df[df["method"] == method_name].sort_values("keep_ratio")
    if sub.empty:
        continue
    ax.plot(
        sub["keep_ratio"], sub["tar_at_far"],
        color=style["color"], ls=style["ls"],
        marker=style["marker"], lw=style["lw"],
        markersize=6, label=method_name,
    )

ax.axhline(baseline_row[0]["tar_at_far"], color="#555", ls=":", lw=1.2, label="Baseline")
ax.scatter(
    cropr_row[0]["keep_ratio"] or 0.5, cropr_row[0]["tar_at_far"],
    marker="*", s=200, zorder=5, color="#6a0dad", label="CropR",
)
ax.set_xlabel("Keep Ratio")
ax.set_ylabel(f"TAR @ FAR={CFG['far_threshold']}")
ax.set_title(f"Open-Set Identification — TAR@FAR ({DATASET_NAME})")
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
plt.tight_layout()
fig.savefig(CFG["results_dir"] / "comparison_tar_at_far.pdf", bbox_inches="tight")
fig.savefig(CFG["results_dir"] / "comparison_tar_at_far.png", dpi=300, bbox_inches="tight")
plt.show()

### 11.2 GFLOPs Comparison Bar Chart (fixed points)

In [ ]:
# Select one representative keep_ratio per method for the bar chart
TARGET_KR = 0.5

bar_data = {}
bar_data["Baseline"] = (baseline_row[0]["f1_macro"], baseline_row[0]["gflops"], baseline_row[0]["ms_per_img"])

for method_name in ["Random", "CLS Attention", "Forecaster", "Forecaster + FT"]:
    sub = df[df["method"] == method_name]
    if sub.empty:
        continue
    row = sub.iloc[(sub["keep_ratio"] - TARGET_KR).abs().argmin()]
    bar_data[f"{method_name}\n(keep≈{TARGET_KR:.0%})"] = (
        row["f1_macro"], row["gflops"], row["ms_per_img"]
    )

bar_data["CropR"] = (cropr_row[0]["f1_macro"], cropr_row[0]["gflops"], cropr_row[0]["ms_per_img"])

best_tome_r = max(tome_records, key=lambda x: x["f1_macro"])
bar_data[f"ToMe\n(r={best_tome_r['r']})"] = (
    best_tome_r["f1_macro"], best_tome_r["gflops"], best_tome_r["ms_per_img"]
)

methods_bar = list(bar_data.keys())
f1s_bar     = [v[0] for v in bar_data.values()]
gflops_bar  = [v[1] if v[1] is not None else 0.0 for v in bar_data.values()]
ms_bar      = [v[2] for v in bar_data.values()]

colors = ["#264653", "#aaaaaa", "#4878cf", "#f4a261", "#e63946", "#6a0dad", "#2a9d8f"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=160)
x = np.arange(len(methods_bar))

for ax, ys, ylabel, title in zip(
    axes,
    [f1s_bar, gflops_bar, ms_bar],
    ["F1 Macro", "GFLOPs / image", "Latency (ms / image)"],
    ["Quality", "Computational Cost", "Speed"],
):
    bars = ax.bar(x, ys, color=colors[:len(x)], edgecolor="white", linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(methods_bar, rotation=25, ha="right", fontsize=8)
    ax.set_ylabel(ylabel)
    ax.set_title(f"{title} ({DATASET_NAME})", fontsize=11)
    ax.grid(axis="y", alpha=0.25)
    for bar, val in zip(bars, ys):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.01,
            f"{val:.3f}" if ylabel == "F1 Macro" else f"{val:.1f}",
            ha="center", va="bottom", fontsize=7,
        )

plt.tight_layout()
fig.savefig(CFG["results_dir"] / "comparison_bar_chart.pdf", bbox_inches="tight")
fig.savefig(CFG["results_dir"] / "comparison_bar_chart.png", dpi=300, bbox_inches="tight")
plt.show()

## 12 · LaTeX Table

In [ ]:
# Build a flat summary for the paper at keep_ratio ≈ 0.1 and ≈ 0.5
paper_rows = []

# Baseline
r = baseline_row[0]
paper_rows.append({
    "Method": "Baseline (No Pruning)",
    "Keep": "100\\%",
    "ACC": r["accuracy"],
    "F1": r["f1_macro"],
    "TAR@FAR": r["tar_at_far"],
    "ms": r["ms_per_img"],
    "GFLOPs": r["gflops"] if r["gflops"] else float("nan"),
})

for method_name in ["Random", "CLS Attention", "Forecaster", "Forecaster + FT"]:
    sub = df[df["method"] == method_name]
    if sub.empty:
        continue
    for kr in [0.5, 0.1]:
        row = sub.iloc[(sub["keep_ratio"] - kr).abs().argmin()]
        paper_rows.append({
            "Method": method_name,
            "Keep": f"{int(kr*100)}\\%",
            "ACC": row["accuracy"],
            "F1": row["f1_macro"],
            "TAR@FAR": row["tar_at_far"],
            "ms": row["ms_per_img"],
            "GFLOPs": row["gflops"] if row["gflops"] else float("nan"),
        })

r = cropr_row[0]
paper_rows.append({
    "Method": "CropR",
    "Keep": "fixed",
    "ACC": r["accuracy"],
    "F1": r["f1_macro"],
    "TAR@FAR": r["tar_at_far"],
    "ms": r["ms_per_img"],
    "GFLOPs": r["gflops"] if r["gflops"] else float("nan"),
})

for r in tome_records:
    paper_rows.append({
        "Method": f"ToMe ($r$={r['r']})",
        "Keep": f"{r['keep_ratio']:.0%}",
        "ACC": r["accuracy"],
        "F1": r["f1_macro"],
        "TAR@FAR": r["tar_at_far"],
        "ms": r["ms_per_img"],
        "GFLOPs": r["gflops"] if r["gflops"] else float("nan"),
    })

df_paper = pd.DataFrame(paper_rows)


def make_latex_table(df_in, caption, label):
    lines = [
        r"\begin{table}[t]",
        r"\centering",
        f"\\caption{{{caption}}}",
        f"\\label{{{label}}}",
        r"\begin{tabular}{llccccc}",
        r"\toprule",
        r"Method & Keep & ACC & F1 Macro & TAR@FAR & ms/img & GFLOPs \\",
        r"\midrule",
    ]
    best_f1 = df_in["F1"].max()
    for _, row in df_in.iterrows():
        f1_str  = f"\\textbf{{{row['F1']:.4f}}}" if row["F1"] == best_f1 else f"{row['F1']:.4f}"
        gf_str  = "--" if np.isnan(row["GFLOPs"]) else f"{row['GFLOPs']:.1f}"
        lines.append(
            f"{row['Method']} & {row['Keep']} & {row['ACC']:.4f} & "
            f"{f1_str} & {row['TAR@FAR']:.4f} & {row['ms']:.2f} & {gf_str} \\\\"
        )
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)


latex = make_latex_table(
    df_paper,
    caption=f"Quantitative comparison of token pruning/reduction methods on {DATASET_NAME}.",
    label="tab:pruning_comparison",
)

latex_path = CFG["results_dir"] / "table_comparison.tex"
latex_path.write_text(latex)
print(f"LaTeX table saved → {latex_path}")
print("\n" + latex)

## 13 · Save All Results to JSON

In [ ]:
all_records_serializable = [
    {k: (float(v) if isinstance(v, (np.floating, np.integer)) else v)
     for k, v in rec.items()}
    for rec in all_records
]
json_path = CFG["results_dir"] / "all_results.json"
with open(json_path, "w") as f:
    json.dump(all_records_serializable, f, indent=2)
print(f"Full results saved → {json_path}")
print(f"\nOutputs in: {CFG['results_dir']}")
for p in sorted(CFG["results_dir"].iterdir()):
    print(f"  {p.name}")